# Needs-Verification Auto-Review Pipeline

End-to-end notebook for verifying augmented NL↔Cypher entity rewrites.

**Pipeline**
1. Load `needs_verification.jsonl` rows from all 3 benchmark datasets
2. For each row: DuckDuckGo search + LLM confidence assessment (via `scripts/walmart_llm.py`)
3. Checkpoint every row to `benchmarks/verification_progress.jsonl` (resume-safe)
4. Build colour-coded Excel for human review
5. Print confidence distribution summary

> **LLM credentials** are configured in `scripts/walmart_llm.py` — no secrets in this notebook.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import re
import sys
import time
import urllib.parse
import warnings
from collections import Counter
from pathlib import Path

import requests

warnings.filterwarnings("ignore")  # suppress SSL verify=False warnings

# ── Paths ──────────────────────────────────────────────────────────────────────
REPO_ROOT  = Path("../").resolve()
BENCHMARKS = REPO_ROOT / "benchmarks"
CHECKPOINT = BENCHMARKS / "verification_progress.jsonl"
OUTPUT_XLSX = BENCHMARKS / "needs_verification_review.xlsx"

DATASETS = [
    "cypherbench_augmented_v2",
    "mindthequery_augmented_v2",
    "zograscope_augmented_v2",
]

# ── LLM module (credentials live in scripts/walmart_llm.py) ───────────────────
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from walmart_llm import ask_llm  # noqa: E402

# ── Run config ─────────────────────────────────────────────────────────────────
LIMIT       = 0      # 0 = process all rows; set e.g. 20 for a smoke test
RESET       = False  # True = wipe checkpoint and restart
EXCEL_ONLY  = False  # True = skip search/LLM; just rebuild Excel from checkpoint
DDG_DELAY   = 2.5    # seconds between DuckDuckGo searches

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"CHECKPOINT: {CHECKPOINT}")
print(f"OUTPUT    : {OUTPUT_XLSX}")

## 2. Load Data

In [ ]:
def load_original_nl() -> dict[str, str]:
    """qid → original NL (before augmentation) from every test.json."""
    mapping: dict[str, str] = {}
    for ds in DATASETS:
        rows = json.loads((BENCHMARKS / ds / "test.json").read_text())
        for r in rows:
            mapping[r["id"]] = r.get("_aug_meta", {}).get("original_nl") or r["nl"]
    return mapping


def load_rows(orig_nl: dict[str, str]) -> list[dict]:
    """Merge needs_verification.jsonl rows from all datasets."""
    rows = []
    for ds in DATASETS:
        with (BENCHMARKS / ds / "needs_verification.jsonl").open() as fh:
            for line in fh:
                r = json.loads(line)
                rows.append({
                    "dataset":      ds,
                    "graph":        r.get("graph", ""),
                    "qid":          r.get("qid", ""),
                    "strategy":     r.get("strategy", ""),
                    "original_nl":  orig_nl.get(r.get("qid", ""), ""),
                    "augmented_nl": r.get("nl", ""),
                    "from":         r.get("from", ""),
                    "to":           r.get("to", ""),
                })
    return rows


orig_nl = load_original_nl()
all_rows = load_rows(orig_nl)
if LIMIT:
    all_rows = all_rows[:LIMIT]

print(f"Loaded {len(all_rows)} rows across {len(DATASETS)} datasets")
print("Sample row:", json.dumps(all_rows[0], indent=2))

## 3. Checkpoint

In [ ]:
if RESET and CHECKPOINT.exists():
    CHECKPOINT.unlink()
    print("Checkpoint cleared.")

def load_checkpoint() -> dict[str, dict]:
    """Return {qid: result_dict} already assessed."""
    done: dict[str, dict] = {}
    if CHECKPOINT.exists():
        with CHECKPOINT.open() as fh:
            for line in fh:
                obj = json.loads(line)
                done[obj["qid"]] = obj
    return done


done = load_checkpoint()
remaining = [r for r in all_rows if r["qid"] not in done]
print(f"{len(all_rows)} total  |  {len(done)} done  |  {len(remaining)} remaining")

## 4. DuckDuckGo Search Helper

In [ ]:
_DDG_URL    = "https://html.duckduckgo.com/html/"
_STRIP_TAGS = re.compile(r"<[^>]+>")
_DDG_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}


def ddg_search(query: str, max_results: int = 4) -> list[dict]:
    """Return up to max_results hits [{title, snippet, url}]. Returns [] on error."""
    try:
        resp = requests.post(
            _DDG_URL, data={"q": query}, headers=_DDG_HEADERS,
            timeout=15, verify=False,
        )
        if resp.status_code != 200:
            return []
        html   = resp.text
        titles = re.findall(r'class="result__title".*?<a[^>]*>(.*?)</a>', html, re.DOTALL)
        snips  = re.findall(r'class="result__snippet">(.*?)</a>',          html, re.DOTALL)
        urls   = re.findall(r'result__url[^>]*>\s*(.*?)\s*<',              html, re.DOTALL)
        results = []
        for t, s, u in zip(titles, snips, urls):
            results.append({
                "title":   _STRIP_TAGS.sub("", t).strip(),
                "snippet": _STRIP_TAGS.sub("", s).strip(),
                "url":     u.strip(),
            })
            if len(results) >= max_results:
                break
        return results
    except Exception:
        return []


# Quick smoke test
hits = ddg_search('"Chicago Bulls" "CHI" nba')
print(f"Search returned {len(hits)} hits")
for h in hits:
    print(f"  [{h['title']}] {h['snippet'][:80]}…")

## 5. LLM Assessment Helper

In [ ]:
_SYSTEM_PROMPT = """\
You are a linguistic fact-checker verifying whether a text rewrite is valid.
You will be given:
  - An original entity string (\"from\")
  - A rewritten entity string (\"to\")
  - The rewrite strategy (abbrev | alias | partial)
  - The graph domain (e.g. nba, flight_accident)
  - DuckDuckGo search results (may be empty)

Confidence rubric:
  high   = multiple independent sources confirm that \"to\" is a standard
           abbreviation/alias/partial-name for \"from\" in this domain, OR
           it is universally well-known (e.g. \"CHI\" = Chicago Bulls in NBA)
  medium = plausible and likely correct but ambiguous; only one weak source,
           or the rewrite is domain-plausible without strong confirmation
  low    = no evidence found, contradicted by search results, or the rewrite
           is implausible/wrong

Respond with ONLY valid JSON (no markdown, no extra text):
{\n  \"confidence\": \"low\" | \"medium\" | \"high\",\n  \"reason\": \"<1-2 sentence evidence summary>\"\n}
"""


def llm_assess(
    from_: str, to: str, strategy: str, graph: str,
    search_results: list[dict],
) -> dict:
    """Return {confidence, reason} from the LLM (via scripts/walmart_llm.py)."""
    if search_results:
        snippets_txt = "\n".join(
            f"- [{r['title']}] {r['snippet']}  ({r['url']})"
            for r in search_results
        )
    else:
        snippets_txt = "(no search results found)"

    prompt = f"""\
from    : {from_}
to      : {to}
strategy: {strategy}
domain  : {graph}

Search results:
{snippets_txt}

Is \"{to}\" a valid {strategy} for \"{from_}\" in the {graph} domain?
"""

    for attempt in range(3):
        try:
            raw  = ask_llm(prompt, system=_SYSTEM_PROMPT, max_tokens=256, temperature=0.0)
            raw  = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            data = json.loads(raw)
            if data.get("confidence") in ("low", "medium", "high") and "reason" in data:
                return data
        except (json.JSONDecodeError, KeyError, RuntimeError):
            if attempt < 2:
                time.sleep(2)

    return {"confidence": "low", "reason": "LLM assessment failed after 3 attempts."}

## 6. Main Processing Loop

> **Resume-safe**: already-assessed rows (in `verification_progress.jsonl`) are skipped.
> Interrupt at any time — re-run this cell to continue.

In [ ]:
if EXCEL_ONLY:
    print("EXCEL_ONLY=True — skipping search/LLM step.")
else:
    todo = [r for r in all_rows if r["qid"] not in done]
    print(f"Processing {len(todo)} remaining rows …\n")

    with CHECKPOINT.open("a") as cp:
        for i, r in enumerate(todo, start=1):
            qid = r["qid"]
            print(
                f"[{i}/{len(todo)}] {r['from']!r} → {r['to']!r}"
                f"  ({r['strategy']}, {r['graph']})",
                end=" … ", flush=True,
            )

            # 1. DuckDuckGo search
            query = f'"{r["from"]}" "{r["to"]}" {r["graph"]}'
            hits  = ddg_search(query)
            urls  = [h["url"] for h in hits]

            # 2. LLM assessment
            assessment = llm_assess(
                from_=r["from"], to=r["to"],
                strategy=r["strategy"], graph=r["graph"],
                search_results=hits,
            )

            result = {
                "qid":           qid,
                "confidence":    assessment["confidence"],
                "reason":        assessment["reason"],
                "evidence_urls": urls,
            }
            done[qid] = result
            cp.write(json.dumps(result) + "\n")
            cp.flush()

            print(assessment["confidence"].upper())
            time.sleep(DDG_DELAY)

    print("\nProcessing complete.")

## 7. Build Excel

In [ ]:
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation


def build_excel(rows: list[dict], results: dict[str, dict], output_path: Path) -> None:
    HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
    HEADER_FONT = Font(bold=True, color="FFFFFF", size=11)
    EVEN_FILL   = PatternFill("solid", fgColor="EBF3FB")
    ODD_FILL    = PatternFill("solid", fgColor="FFFFFF")
    CONF_COLORS = {
        "high":   PatternFill("solid", fgColor="C6EFCE"),  # green
        "medium": PatternFill("solid", fgColor="FFEB9C"),  # yellow
        "low":    PatternFill("solid", fgColor="FFC7CE"),  # red
    }
    WRAP   = Alignment(wrap_text=True, vertical="top")
    CENTER = Alignment(horizontal="center", vertical="top", wrap_text=True)
    BORDER = Border(
        bottom=Side(style="thin", color="BDD7EE"),
        right=Side(style="thin", color="BDD7EE"),
    )

    COLUMNS = [
        ("dataset",            22),
        ("graph",              16),
        ("qid",                36),
        ("strategy",           12),
        ("original_nl",        52),
        ("augmented_nl",       52),
        ("from",               28),
        ("to",                 24),
        ("auto_confidence",    18),
        ("auto_reason",        60),
        ("evidence_urls",      40),
        ("google_search",      18),
        ("human_verification", 22),
        ("notes",              40),
    ]

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Verification"

    headers = [c for c, _ in COLUMNS]
    for col_idx, (header, width) in enumerate(COLUMNS, start=1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font      = HEADER_FONT
        cell.fill      = HEADER_FILL
        cell.alignment = CENTER
        ws.column_dimensions[get_column_letter(col_idx)].width = width
    ws.row_dimensions[1].height = 24

    CONF_COL   = headers.index("auto_confidence")    + 1
    GOOGLE_COL = headers.index("google_search")      + 1
    VERIFY_COL = headers.index("human_verification") + 1

    for row_idx, r in enumerate(rows, start=2):
        base_fill = EVEN_FILL if row_idx % 2 == 0 else ODD_FILL
        res  = results.get(r["qid"], {})
        conf = res.get("confidence", "")
        urls = res.get("evidence_urls", [])

        values = [
            r["dataset"], r["graph"], r["qid"], r["strategy"],
            r["original_nl"], r["augmented_nl"], r["from"], r["to"],
            conf,
            res.get("reason", ""),
            " | ".join(urls),
            "🔍 Search",
            "",   # human_verification — blank
            "",   # notes — blank
        ]

        for col_idx, value in enumerate(values, start=1):
            cell            = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.fill       = base_fill
            cell.border     = BORDER
            cell.alignment  = WRAP

        # Confidence — colour-coded
        conf_cell = ws.cell(row=row_idx, column=CONF_COL)
        if conf in CONF_COLORS:
            conf_cell.fill = CONF_COLORS[conf]
        conf_cell.alignment = CENTER

        # Google search hyperlink
        q     = urllib.parse.quote_plus(f'"{r["from"]}" "{r["to"]}" {r["graph"]}')
        gcell = ws.cell(row=row_idx, column=GOOGLE_COL)
        gcell.hyperlink = f"https://www.google.com/search?q={q}"
        gcell.value     = "🔍 Search"
        gcell.font      = Font(color="0563C1", underline="single")
        gcell.alignment = CENTER

        ws.row_dimensions[row_idx].height = 52

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    dv = DataValidation(
        type="list",
        formula1='"CORRECT,INCORRECT,UNSURE"',
        allow_blank=True,
        showDropDown=False,
        showErrorMessage=True,
        errorTitle="Invalid value",
        error="Choose CORRECT, INCORRECT, or UNSURE.",
    )
    ws.add_data_validation(dv)
    last = len(rows) + 1
    dv.add(f"{get_column_letter(VERIFY_COL)}2:{get_column_letter(VERIFY_COL)}{last}")

    wb.save(output_path)
    print(f"Excel written → {output_path}")


build_excel(all_rows, done, OUTPUT_XLSX)

## 8. Summary

In [ ]:
rows_done = list(done.values())
total     = len(rows_done)
counts    = Counter(r["confidence"] for r in rows_done)


def pct(n: int) -> str:
    return f"{n / total * 100:5.1f}%" if total else "  n/a"


print(f"{'─' * 50}")
print(f"  Total assessed      : {total}")
print(f"    HIGH  (green)     : {counts['high']:>4}   ({pct(counts['high'])})")
print(f"    MEDIUM (yellow)   : {counts['medium']:>4}   ({pct(counts['medium'])})")
print(f"    LOW   (red)       : {counts['low']:>4}   ({pct(counts['low'])})")
print(f"{'─' * 50}")
print(f"  Checkpoint : {CHECKPOINT.relative_to(REPO_ROOT)}")
print(f"  Excel      : {OUTPUT_XLSX.relative_to(REPO_ROOT)}", end="")
if OUTPUT_XLSX.exists():
    print(f"  ({OUTPUT_XLSX.stat().st_size / 1024:.1f} KB)")
else:
    print()
print()
print("  Next: open the Excel and fill in 'human_verification'.")
print("        Prioritise rows marked LOW — highest-confidence mismatches.")